In [1]:
#SAR Recommendation System handles cold item and semi cold users.
#SAR recommends items that are most similar to the ones that the users
# has an existing affinity for. 
#Two items are similar if the user has interacted with one item are also
#likely to have interacted with other. 
#A user has affinity to an item if they have interacted with it in the past


In [2]:
import logging
import numpy as np 
import pandas as pd
from sklearn.preprocessing import minmax_scale

from recommenders.utils.timer import Timer
from recommenders.datasets import movielens
from recommenders.utils.python_utils import binarize 
from recommenders.datasets.python_splitters import python_stratified_split
from recommenders.models.sar import SAR
from recommenders.evaluation.python_evaluation import (
    map_at_k,ndcg_at_k,precision_at_k,recall_at_k,rmse,mae,logloss,rsquared,exp_var
)

c:\Users\muzic\AppData\Local\Programs\Python\Python310\lib\site-packages\pandera\_pandas_deprecated.py:154: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


In [3]:
#top k items to recommend
TOP_K = 10 

#select movielens datasize: 100k 
MOVIELENS_DATA_SIZE = "100k"

#other data settings
USER_COL = "userID"
ITEM_COL = "itemID"
RATING_COL = "rating"
TIMESTAMP_COL = "timestamp"
PREDICTION_COL = "prediction"

#Train test split ratio
SPLIT_RATIO = 0.75

#Model settings
SIMILARITY_TYPE = "jaccard"
TIME_DECAY = 30 #number of days until the weight of the ratings are decayed by 1/2

SEED = 42

logging.basicConfig(level=logging.DEBUG, format="%(levelname)-8s %(message)s")

In [4]:
data = movielens.load_pandas_df(
    size = MOVIELENS_DATA_SIZE
)

#convert the float precision to 32 bit to save memory
data[RATING_COL] = data[RATING_COL].astype(np.float32)

data.head()

DEBUG    Starting new HTTPS connection (1): files.grouplens.org:443
DEBUG    https://files.grouplens.org:443 "GET /datasets/movielens/ml-100k.zip HTTP/1.1" 200 4924029
INFO     Downloading https://files.grouplens.org/datasets/movielens/ml-100k.zip
100%|██████████| 4.81k/4.81k [00:00<00:00, 9.45kKB/s]


,userID,itemID,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [5]:
#split the data into train and test set 

train, test = python_stratified_split(data, ratio = SPLIT_RATIO, filter_by="user",col_user = USER_COL, seed = SEED)

In [6]:
print("""
    Train:
    Total Ratings : {train_total}
    Unique Users : {train_users}
    Unique Items : {train_items}
      
    Test:
    Total Ratings : {test_total}
    Unique Users : {test_users}
    Unique Items : {test_items}
    """.format(
        train_total = len(train),
        train_users = len(train[USER_COL].unique()),
        train_items = len(train[ITEM_COL].unique()),
        test_total = len(test),
        test_users = len(test[USER_COL].unique()),
        test_items = len(test[ITEM_COL].unique())
    )
)


    Train:
    Total Ratings : 74992
    Unique Users : 943
    Unique Items : 1653
      
    Test:
    Total Ratings : 25008
    Unique Users : 943
    Unique Items : 1444
    


In [7]:
#instantiate the SAR model 

model = SAR(
    col_user = USER_COL,
    col_item = ITEM_COL,
    col_rating = RATING_COL,
    col_timestamp = TIMESTAMP_COL,
    similarity_type=SIMILARITY_TYPE,
    time_decay_coefficient = 30,
    timedecay_formula = True,
    normalize=True
)

In [8]:
#training the SAR model

with Timer() as train_time:
    model.fit(train)

print(f"Took{train_time.interval} seconds to train the model")

INFO     Collecting user affinity matrix
INFO     Calculating time-decayed affinities
INFO     Creating index columns
INFO     Calculating normalization factors
INFO     Building user affinity sparse matrix
INFO     Calculating item co-occurrence
INFO     Calculating item similarity
INFO     Using jaccard based similarity
INFO     Done training


Took0.755549699999392 seconds to train the model


In [10]:
#test the model
with Timer() as test_time:
    top_k = model.recommend_k_items(test,top_k = TOP_K,remove_seen=True)
print(f"Took {test_time.interval} seconds to test the model")

INFO     Calculating recommendation scores
INFO     Removing seen items


Took 0.39507930004037917 seconds to test the model


In [11]:
top_k.head()

,userID,itemID,prediction
0,1,433,2.910697
1,1,204,2.906224
2,1,403,2.906136
3,1,174,2.870639
4,1,70,2.863253


In [12]:
#eval how well the SAR model performed using the evaluation metrics


In [13]:
#Ranking metrics
eval_map = map_at_k(test,top_k,col_user=USER_COL,col_item = ITEM_COL,col_rating = RATING_COL, k =TOP_K)
eval_ndcg = ndcg_at_k(test,top_k,col_user=USER_COL,col_item = ITEM_COL,col_rating = RATING_COL, k =TOP_K)
eval_precision = precision_at_k(test,top_k,col_user=USER_COL,col_item = ITEM_COL,col_rating = RATING_COL, k =TOP_K)
eval_recall = recall_at_k(test,top_k,col_user=USER_COL,col_item = ITEM_COL,col_rating = RATING_COL, k =TOP_K)

In [14]:
#Rating metrics

eval_rmse = rmse(test,top_k,col_user=USER_COL,col_item = ITEM_COL, col_rating = RATING_COL)
eval_mae = mae(test,top_k,col_user=USER_COL,col_item = ITEM_COL, col_rating = RATING_COL)
eval_rsquared = rsquared(test,top_k,col_user=USER_COL,col_item = ITEM_COL, col_rating = RATING_COL)
eval_exp_var = exp_var(test,top_k,col_user=USER_COL,col_item = ITEM_COL, col_rating = RATING_COL)

In [16]:
positivity_threshold = 2
test_bin = test.copy()
test_bin[RATING_COL] = binarize(test_bin[RATING_COL], positivity_threshold)

top_k_prob = top_k.copy()
top_k_prob[PREDICTION_COL] = minmax_scale(top_k_prob[PREDICTION_COL].astype(float))

eval_logloss = logloss(
    test_bin, top_k_prob , col_user=USER_COL, col_item = ITEM_COL, col_rating = RATING_COL, col_prediction = PREDICTION_COL
)

In [17]:
print("Model:\t\tSAR",
      "TOP K:\t\t\t%d" % TOP_K,
      "NDCG:\t\t%f" % eval_ndcg,
      "Precision:\t%f" % eval_precision,
      "Recall:\t\t%f" % eval_recall,
      "RMSE:\t\t%f" % eval_rmse,
      "MAE:\t\t%f" % eval_mae,
      "R2:\t\t%f" % eval_rsquared,
      "Exp Var:\t%f" % eval_exp_var,
      "Log Loss:\t%f" % eval_logloss,
      )


Model:		SAR TOP K:			10 NDCG:		0.379533 Precision:	0.331071 Recall:		0.176837 RMSE:		1.229246 MAE:		1.033912 R2:		-0.511334 Exp Var:	0.098494 Log Loss:	0.569153


In [18]:
#lets look result for a specific user
user_id = 200

user_input = pd.DataFrame(dict(userID=[user_id]))

In [19]:
ground_truth = test[test[USER_COL] == user_id]
prediction = model.recommend_k_items(user_input,remove_seen=True)

df=pd.merge(ground_truth, prediction, on=[USER_COL, ITEM_COL], how="left").sort_values(by=PREDICTION_COL,ascending=False)
df

INFO     Calculating recommendation scores
INFO     Removing seen items


,userID,itemID,rating,timestamp,prediction
46,200,423,5.0,884129275,3.815873
34,200,96,5.0,884129409,3.733348
43,200,56,4.0,884128858,3.640387
0,200,234,4.0,884129381,3.610741
1,200,235,2.0,884128065,NaN
2,200,580,2.0,884130114,NaN
3,200,118,4.0,876042299,NaN
4,200,112,3.0,884127370,NaN
5,200,15,4.0,884127745,NaN
6,200,121,5.0,876042268,NaN
